In [2]:
# General imports
import pandas as pd
import numpy as np
import requests
import os
import time
import json

In [3]:
# Report dict to map values for API
report_map = {
    'GIA': 0,
    'GCAL': 5,
    'IGI': 2
}

In [4]:
# Color dict to map values for API
color_map = {
    'K': 8,
    'J': 7,
    'I': 6,
    'H': 5,
    'G': 4,
    'F': 3,
    'E': 2,
    'D': 1
}

In [5]:
# Clarity dict to map values for API
clarity_map = {
    'SI2': 8,
    'SI1': 7,
    'VS2': 6,
    'VS1': 5,
    'VVS2': 4,
    'VVS1': 3,
    'IF': 2,
    'FL': 1
}

In [6]:
# Cut dict to map values for API
# We replace Excellent by Premium
cut_map = {
    'Good': 3,
    'Very Good': 2,
    'Premium': 1,
    'Ideal': 0,
    'Super Ideal': -1
}


In [7]:
# Shape dict to map values for API
shape_map = {
    'Round': 1,
    'Oval': 6,
    'Cushion': 10,
    'Emerald': 4,
    'Princess': 2,
    'Radiant': 8, 
    'Pear': 5,
    'Marquise': 3,
    'Asscher': 9,
    'Heart': 7
}

In [8]:
# Define the JSON payload to be posted to the API
def get_payload(cut, color, clarity, carat, shape, report, lab_grown):

    payload = {
        "diamond": {
            "caratMin": carat,
            "caratMax": carat,
            "certificateLabs": [report],
            "clarityMin": clarity,
            "clarityMax": clarity,
            "colorMin": color,
            "colorMax": color,
            "crownAngleMin": 23,
            "crownAngleMax": 40,
            "cutMin": cut,
            "cutMax": cut,
            "dealScoreRatings": [],
            "depthPercentageMin": 0,
            "depthPercentageMax": 100,
            "fluorescenceMin": 0,
            "fluorescenceMax": 4,
            "girdleThicknessPercentageMin": 1.5,
            "girdleThicknessPercentageMax": 7,
            "girdleThicknessMin": 1,
            "girdleThicknessMax": 8,
            "heightMin": 2,
            "heightMax": 12,
            "lengthToWidthRatioMin": 1,
            "lengthToWidthRatioMax": 2.75,
            "lengthMin": 3,
            "lengthMax": 20,
            "hasMedia": False,
            "pairSearch": False,
            "pavilionAngleMin": 38,
            "pavilionAngleMax": 43,
            "polishMin": 1,
            "polishMax": 3,
            "pricePerCaratMin": 0,
            "pricePerCaratMax": 50000,
            "priceMin": 300,
            "priceMax": 20000,
            "qualityScoreRankings": [],
            #"shapes": [1, 6, 10, 4, 2, 8, 5, 3, 9, 7],
            "shapes": [shape], 
            "shippingDays": -1,
            "symmetryMin": 1,
            "symmetryMax": 3,
            "tableWidthPercentageMin": 0,
            "tableWidthPercentageMax": 100,
            "isLabGrown": lab_grown,
            "widthMin": 3,
            "widthMax": 20
        },
        "retailer": {
            "distance": None,
            "postalCode": None,
            "localRetailers": [],
            "retailers": [],
            "showOnline": True,
            "showLocal": True,
            "features": []
        },
        "setting": {
            "metals": [],
            "priceMax": 20000,
            "priceMin": 300,
            "styles": []
        },
        "orderBy": "best-value",
        "orderDirection": True,
        "prioritizeB2CVideos": False,
        "customSearchFilter": ""
    }

    return payload

In [9]:
def get_request(payload):

    # Define the endpoint URL (update with the correct API endpoint)
    url = "https://webapi.rarecarat.com/diamonds2"  # Replace with the actual API URL

    # Define headers
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0"  # Mimic a browser request to avoid being blocked
    }   

    # Send the POST request
    response = requests.post(url, headers=headers, json=payload)
    print("Status Code:", response.status_code)
    
    return response

In [10]:
def process_results(price):
    
    df = pd.DataFrame(price)
    print(f'Total records: {len(df.index)}')

    #df_max = df.loc[df.groupby('retailer')['retail_price'].idxmax()]
    #print(f'Total filtered records: {len(df_max.index)}')
    
    # Function to get the index where retail_price is closest to the mean per group
    idx_mean = df.groupby('retailer')['retail_price'].apply(lambda x: (x - x.mean()).abs().idxmin())

    # Get rows corresponding to idx_mean (like an 'idxmean()' function)
    df_avg = df.loc[idx_mean]  

    print(f'Total filtered records: {len(df_avg.index)}')
   

    return df_avg

In [19]:
def get_data(diamond_data, csv_file, lab_grown=True, index=0 ):

    diamond_dict = diamond_data.to_dict(orient="records")

    price_csv_file = csv_file
    
    lab_grown = lab_grown
    
    index = index

    price_df = pd.DataFrame()

    delay = 15

    for i in diamond_dict[index:]:
        print(f'Processing vec: {i}')
        price = []
        cut = cut_map[i['cut']]
        color = color_map[i['color']]
        clarity = clarity_map[i['clarity']]
        carat = i['carat']
        shape = shape_map[i['shape']]
        report = report_map[i['report']]
        payload = get_payload(cut, color, clarity, carat, shape, report, lab_grown)
        #print(payload)
        try:
            resp = get_request(payload)
            resp.raise_for_status()  # Raise HTTPError for bad status codes (4xx or 5xx)
        except requests.exceptions.HTTPError as errh:
            print(f"HTTP Error: {errh}")
            time.sleep(60)
        except requests.exceptions.ConnectionError as errc:
            print(f"Connection Error: {errc}")
            time.sleep(60)
        except requests.exceptions.Timeout as errt:
            print(f"Timeout Error: {errt}")
            time.sleep(60)
        except requests.exceptions.RequestException as err:
            print(f"Something went wrong: {err}")
            time.sleep(60)
        else:
            resp_json = resp.json()
            for diamond in resp_json['diamonds']:
                if 'price' in diamond:
                    retail_price = diamond['price']
                else:
                    retail_price = round(diamond['pricePerCarat'] * carat,2)
                if 'discountPercentage' in diamond:
                    discount = diamond['discountPercentage']
                else:
                    discount = None
                if 'qualityMaxScore' in diamond:
                    max_score = diamond['qualityMaxScore']
                else:
                    max_score = None
                if 'qualityScore' in diamond:
                    score = diamond['qualityScore']
                else:
                    score = None
                #print(payload)
                #print(f"[{i['cut']}, {i['color']}, {i['clarity']}, {carat}]: dataset_price: {i['price']}, retailer: {diamond['retailer']['name']}, retailer_price: {diamond['price']} discount: {diamond['discountPercentage']} hidden_price: {diamond['hiddenPredictedPriceV2']}")
                #print(json.dumps(diamond, indent=4))
                # Full version
                #price.append({"index": index, "retailer": diamond['retailer']['name'], "lab" : diamond['certificateLab'], "shape": diamond['shape'], "polish": diamond['polish'], "symmetry": diamond['symmetry'], "fluorescence": diamond['fluorescence'], "discount_%": discount, "retail_price": retail_price, "price_carat": diamond['pricePerCarat'], "wire_price": diamond['wirePrice'], "fair_price_diif": diamond['fairPriceDifference'], "hidden_price": diamond['hiddenPredictedPriceV2'], "qualityScore": score, "qualityMaxScore": max_score})
                price.append({"index": index, "retailer": diamond['retailer']['name'], "lab" : diamond['certificateLab'], "shape": diamond['shape'], "polish": diamond['polish'], "symmetry": diamond['symmetry'], "fluorescence": diamond['fluorescence'], "discount_%": discount, "retail_price": retail_price, "price_carat": round(diamond['pricePerCarat'],2), "wire_price": diamond['wirePrice'], "fair_price_diif": diamond['fairPriceDifference'], "hidden_price": diamond['hiddenPredictedPriceV2'], "qualityScore": score, "qualityMaxScore": max_score}) 
            if len(price) > 0:
                price_retail_df = process_results(price)
                price_retail_df = price_retail_df.assign(**i)
                price_retail_df.drop_duplicates(inplace=True)
                
                if os.path.exists(price_csv_file):
                    price_df = pd.read_csv(price_csv_file)  # Load existing data  
                    price_df = pd.concat([price_df, price_retail_df], ignore_index=True)
                    price_df.to_csv(price_csv_file, index=False)
                else:
                    price_df = pd.concat([price_df, price_retail_df], ignore_index=True)
                    price_df.to_csv(price_csv_file, index=False)
            else:
                print("No records found")


        index += 1    
        time.sleep(delay)

Instructions

1 - Make sure the notebook and the csv files are in the same directory

2 - Run all the previous cells and STOP HERE TO READ

3 - Select your corresponding cells. You have been assigned 2 files: diamond_lab_X.csv and diamond_natural_X.csv

4 - Uncoment the first 2 lines in the first cell to process the diamond_lab_X. 

5 - Run the cell. Go do your stuff and come back in 10 hrs 😉

6 - Uncomment the second 2 lines in the next cell to process the diamond_natural_X

7 - Run the cell. Go do yopur stuff and come back in 10 hrs 😉

### Carlos 

In [ ]:
# Carlos - Step 1
# diamond_lab_1 = pd.read_csv("diamond_lab_1.csv")
# get_data(diamond_lab_1, "processed_diamond_lab_1.csv", lab_grown=True, index=0)


In [21]:
# Carlos - Step 2
# diamond_natural_1 = pd.read_csv("diamond_natural_1.csv")
# get_data(diamond_natural_1, "processed_diamond_natural_1.csv", lab_grown=False, index=0)

### Lucia

In [22]:

# Lucia - Step 1
# diamond_lab_2 = pd.read_csv("diamond_lab_2.csv")
# get_data(diamond_lab_2, "processed_diamond_lab_2.csv", lab_grown=True, index=0)

In [23]:
# Lucia - Step 2
# diamond_natural_2 = pd.read_csv("diamond_natural_2.csv")
# get_data(diamond_natural_2, "processed_diamond_natural_2.csv", lab_grown=False, index=0)

### Manini

In [ ]:
# Manini - Spet 1
# diamond_lab_3 = pd.read_csv("diamond_lab_3.csv")
# get_data(diamond_lab_3, "processed_diamond_lab_3.csv", lab_grown=True, index=0)


In [ ]:
# Manini - Step 2
# diamond_natural_3 = pd.read_csv("diamond_natural_3.csv")
# get_data(diamond_natural_3, "processed_diamond_natural_3.csv", lab_grown=False, index=0)

### Ricardo

In [26]:
# Ricardo - Step 1
# diamond_lab_4 = pd.read_csv("diamond_lab_4.csv")
# get_data(diamond_lab_4, "processed_diamond_lab_4.csv", lab_grown=True, index=0)

In [27]:
# Ricardo - Step 2
# diamond_natural_4 = pd.read_csv("diamond_natural_4.csv")
# get_data(diamond_natural_4, "processed_diamond_natural_4.csv", lab_grown=False, index=0)